# 3-1절 연습 문제 풀이

이 노트북은 3-1절 연습 문제(3-1, 3-2)의 풀이 예시다. 두 문제 모두 손으로 푸는 문제이므로 해설을 먼저 싣고,
답을 확인하는 코드를 덧붙였다.

- 본문 예제 코드는 `code_examples/ch03/` 아래 예제 노트북을 참고한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import torch

## 연습 문제 3-1

> [그림 3-6]을 참고해 다음 조건의 다층 퍼셉트론의 구조를 종이에 그려 보자. 층 사이의 연결선은 자세히 그리지 않아도 된다.
> - 3x2 크기의 이미지를 입력받고, 2x3 크기의 이미지를 출력하는 다층 퍼셉트론(이미지의 각 픽셀은 0과 1 사이의 값 하나로 표현된다.)
> - 뉴런의 수가 각각 열 개, 여덟 개인 은닉층 두 개를 포함
>
> 그린 구조를 보고 각 계층의 가중치 파라미터와 편향 파라미터의 수, 그리고 모델 전체의 파라미터 수를 계산한 후,
> 다층 퍼셉트론의 구조와 파라미터 수의 관계를 설명해 보자.
>
> 힌트: 입력과 출력 이미지의 형태는 무시하고 픽셀 수만 생각한다.

### 손으로 푼 풀이

힌트대로 이미지의 형태는 무시하고 픽셀 수만 세면 입력은 3 × 2 = 6개, 출력은 2 × 3 = 6개다.
따라서 구조는 **6 → 10 → 8 → 6**이다. 입력층은 파라미터가 없는 가상의 계층이므로 파라미터를 가진 계층은 세 개다.

| 계층 | 입력 수 | 뉴런 수 | 가중치 | 편향 |
|---|---|---|---|---|
| 첫 번째 은닉층 | 6 | 10 | 6 × 10 = 60 | 10 |
| 두 번째 은닉층 | 10 | 8 | 10 × 8 = 80 | 8 |
| 출력층 | 8 | 6 | 8 × 6 = 48 | 6 |
| **합계** | | | **188** | **24** |

전체 파라미터는 188 + 24 = **212개**다.

**구조와 파라미터 수의 관계**는 표에서 그대로 읽힌다.
어떤 계층의 가중치 수는 `이전 계층의 출력 수 × 그 계층의 뉴런 수`이고, 편향 수는 `그 계층의 뉴런 수`와 같다.
완전 연결 계층이라 모든 입력이 모든 뉴런에 연결되기 때문이다.
따라서 뉴런을 하나 늘리면 파라미터는 `이전 계층의 출력 수 + 1`만큼 늘고, 계층을 하나 끼워 넣으면
앞뒤 계층의 크기를 곱한 만큼 늘어난다. **뉴런 수는 파라미터를 더하기로, 계층 수는 곱하기로 늘린다**고 볼 수 있다.
입출력이 각각 픽셀 여섯 개뿐인 작은 모델인데도 파라미터가 212개나 되는 이유다.

In [2]:
STRUCTURE = [6, 10, 8, 6]     # 입력 6 -> 은닉 10 -> 은닉 8 -> 출력 6

total_w = total_b = 0
print(f'{"계층":>14} {"입력 수":>7} {"뉴런 수":>7} {"가중치":>7} {"편향":>5}')
print('-' * 46)
names = ['첫 번째 은닉층', '두 번째 은닉층', '출력층']
for name, in_features, out_features in zip(names, STRUCTURE[:-1], STRUCTURE[1:]):
    w, b = in_features * out_features, out_features
    total_w, total_b = total_w + w, total_b + b
    print(f'{name:>14} {in_features:7d} {out_features:7d} {w:7d} {b:5d}')
print('-' * 46)
print(f'{"합계":>14} {"":>7} {"":>7} {total_w:7d} {total_b:5d}')
print(f'\n전체 파라미터 수: {total_w + total_b}개')

            계층    입력 수    뉴런 수     가중치    편향
----------------------------------------------
      첫 번째 은닉층       6      10      60    10
      두 번째 은닉층      10       8      80     8
           출력층       8       6      48     6
----------------------------------------------
            합계                     188    24

전체 파라미터 수: 212개


In [3]:
# 파이토치 모델로 만들어 같은 값이 나오는지 확인
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(6, 10), nn.Sigmoid(),
    nn.Linear(10, 8), nn.Sigmoid(),
    nn.Linear(8, 6), nn.Sigmoid(),
)
counted = sum(p.numel() for p in model.parameters())
print(f'파이토치 모델의 파라미터 수: {counted}개')

파이토치 모델의 파라미터 수: 212개


### 문제 검토

- **적절성: 적합.** 그리기와 계산을 한 문제에 묶어, 구조를 그려 봐야 파라미터 수를 셀 수 있도록 설계했다.
  입출력이 픽셀 여섯 개뿐인 작은 모델인데도 파라미터가 212개나 나오는 점이 마지막 질문(구조와 파라미터 수의 관계)의
  답을 체감하게 한다. 은닉층을 10, 8로 잡은 것도 그 인상을 만드는 데 기여한다.
- **[검토] 그리는 부담.** 6 + 10 + 8 + 6 = 30개의 동그라미를 그려야 한다. 연결선을 생략해도 된다는 안내는 있지만,
  뉴런이 많은 계층을 몇 개만 그리고 가운데를 점으로 줄여도 된다는 안내가 더 있으면 부담이 줄어든다.
  신경망 그림의 일반적인 관례이기도 하다.
- **[검토] '층'과 '계층'.** 지문의 "층 사이의 연결선"은 본문 용어인 '계층'과 다르다(1단계 보고서 항목 1).
- **[검토] 3x2 → 2x3.** 픽셀 수가 6개로 같아 힌트의 취지를 살리는 장치이지만, 전치(transpose)처럼 읽힐 수 있다.
  구조만 묻는 문제라 심각하지 않으므로 그대로 두어도 된다.

## 연습 문제 3-2

> 다음 [그림 3-7]과 같은 다층 퍼셉트론에서 입력 (x1, x2)가 (-1, -1), (-1, 1), (1, -1), (1, 1)일 때의 출력값을
> 각각 계산해 보자. 단, 모든 뉴런의 편향은 1로, 가중치는 각 뉴런에 표시된 숫자를 사용하며 계산 편의를 위해
> 활성화 함수는 입력한 값을 그대로 출력하는 항등 함수(σ(x)=x)로 가정한다.

### 손으로 푼 풀이

그림에서 읽은 파라미터는 다음과 같다. 은닉층 뉴런 세 개, 출력층 뉴런 두 개이며 편향은 모두 1이다.

- 은닉 뉴런 1: 가중치 (1, 1) → h1 = x1 + x2 + 1
- 은닉 뉴런 2: 가중치 (-1, 1) → h2 = -x1 + x2 + 1
- 은닉 뉴런 3: 가중치 (2, 0) → h3 = 2x1 + 1
- 출력 뉴런 1: 가중치 (1, 1, -1) → y1 = h1 + h2 - h3 + 1
- 출력 뉴런 2: 가중치 (-1, 1, 2) → y2 = -h1 + h2 + 2h3 + 1

활성화 함수가 항등 함수이므로 가중합이 그대로 다음 계층의 입력이 된다. 네 입력에 대해 계산하면 이렇다.

| 입력 (x1, x2) | h1 | h2 | h3 | **y1** | **y2** |
|---|---|---|---|---|---|
| (-1, -1) | -1 | 1 | -1 | **2** | **1** |
| (-1, 1) | 1 | 3 | -1 | **6** | **1** |
| (1, -1) | 1 | -1 | 3 | **-2** | **5** |
| (1, 1) | 3 | 1 | 3 | **2** | **5** |

계산을 마치고 나면 한 가지가 눈에 들어온다. 활성화 함수가 항등 함수이므로 이 모델 전체는 결국
입력의 일차식으로 정리된다. 실제로 y2 = -h1 + h2 + 2h3 + 1 = 2x1 + 3 으로, x2가 사라진 일차식이다.
본문 각주 8이 설명한 "활성화 함수가 없으면 여러 계층이 하나의 선형 계층으로 합쳐진다"를 직접 확인하는 셈이다.

In [4]:
# 그림 3-7에서 읽은 파라미터
W_hidden = torch.tensor([[1., 1.], [-1., 1.], [2., 0.]])   # 은닉 뉴런 3개의 가중치
W_output = torch.tensor([[1., 1., -1.], [-1., 1., 2.]])    # 출력 뉴런 2개의 가중치
BIAS = 1.                                                  # 모든 뉴런의 편향

X = torch.tensor([[-1., -1.], [-1., 1.], [1., -1.], [1., 1.]])

# 항등 활성화 함수이므로 가중합이 그대로 다음 계층의 입력이 된다
H = X @ W_hidden.T + BIAS
Y = H @ W_output.T + BIAS

print(f'{"입력":>10} {"은닉층 출력":>18} {"y1":>6} {"y2":>6}')
print('-' * 46)
for x, h, y in zip(X.tolist(), H.tolist(), Y.tolist()):
    xs = f'({int(x[0])}, {int(x[1])})'
    hs = f'({int(h[0])}, {int(h[1])}, {int(h[2])})'
    print(f'{xs:>10} {hs:>18} {int(y[0]):6d} {int(y[1]):6d}')

        입력             은닉층 출력     y1     y2
----------------------------------------------
  (-1, -1)        (-1, 1, -1)      2      1
   (-1, 1)         (1, 3, -1)      6      1
   (1, -1)         (1, -1, 3)     -2      5
    (1, 1)          (3, 1, 3)      2      5


In [5]:
# 활성화 함수가 항등 함수이면 모델 전체가 하나의 선형 계층으로 합쳐진다 (본문 각주 8)
W_combined = W_output @ W_hidden                       # 두 가중치 행렬의 곱
b_combined = W_output @ torch.full((3,), BIAS) + BIAS  # 편향도 하나로 합쳐진다
print(f'합쳐진 가중치:\n{W_combined}')
print(f'합쳐진 편향: {b_combined}')
print(f'\n합쳐진 선형 계층의 출력:\n{X @ W_combined.T + b_combined}')

합쳐진 가중치:
tensor([[-2.,  2.],
        [ 2.,  0.]])
합쳐진 편향: tensor([2., 3.])

합쳐진 선형 계층의 출력:
tensor([[ 2.,  1.],
        [ 6.,  1.],
        [-2.,  5.],
        [ 2.,  5.]])


### 문제 검토

- **적절성: 적합.** 순전파를 손으로 한 번 끝까지 따라가게 하는 문제다. 은닉층이 있는 모델에서 계산이 어떻게
  이어지는지 직접 겪어 보는 것이 3-1절의 마무리로 알맞다. 정수만 나오도록 가중치를 고른 것도 좋다.
- **[검토] 항등 함수를 쓴 결과가 오히려 좋은 관찰거리다.** 계산을 마치면 이 모델이 입력의 일차식으로 정리된다.
  실제로 y2는 2x1 + 3이 되어 x2가 사라진다. 본문 각주 8이 설명한 내용을 손으로 확인할 수 있는 자리인데
  지문이 이를 가리키지 않는다. 한 구절이면 문제의 가치가 크게 올라간다.
- **[중요] 그림 3-7의 입력층 표현이 모호하다.** 그림에서 x1, x2 다음에 **가중치가 표시되지 않은 큰 원 두 개**가
  은닉층 뉴런과 같은 모양으로 그려져 있다. 본문 p6은 입력층을 "사실 존재하지 않는 가상의 계층"이라고 설명하는데,
  그림에서는 뉴런처럼 보인다. 지문이 "모든 뉴런의 편향은 1"이라고 했으므로 독자는 이 원들에도 편향 1이 있는지,
  그렇다면 가중치는 무엇인지 고민하게 된다. 그림에서 이 원들을 빼거나 다른 모양으로 그리는 편이 좋다.

**윤문안**

> **3-2**. 다음 [그림 3-7]과 같은 다층 퍼셉트론에서 입력 (x1, x2)가 (-1, -1), (-1, 1), (1, -1), (1, 1)일 때의
> 출력값을 각각 계산해 보자. 단, 모든 뉴런의 편향은 1로, 가중치는 각 뉴런에 표시된 숫자를 사용하며 계산 편의를 위해
> 활성화 함수는 입력한 값을 그대로 출력하는 항등 함수(σ(x)=x)로 가정한다.
> 계산을 마친 뒤, 출력값 y1과 y2를 x1, x2에 대한 식으로 정리해 보고 그 결과가 무엇을 뜻하는지 생각해 보자.